In [9]:
import numpy as np
from numba import njit, prange

np.random.seed(11)
arr = np.random.rand(1000, 1000)


def calculate_distances1(arr):
    m = arr.shape[0]
    n = arr.shape[1]
    dist_arr = np.zeros((m, m))
    for i in range(m):
        for j in range(i):
            v = 0.0
            for k in range(n):
                v += abs(arr[i, k] - arr[j, k])
            dist_arr[i, j] = v
            dist_arr[j, i] = v
    return dist_arr
# 1.03 s ± 14.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


def calculate_distances2(arr):
    m = arr.shape[0]
    dist_arr = np.zeros((m, m))
    for i in range(m):
        for j in range(i):
            dist_arr[i, j] = np.linalg.norm(arr[i] - arr[j], 1)
            dist_arr[j, i] = dist_arr[i, j]
    return dist_arr
# 1.03 s ± 14.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


@njit(cache=True)
def calculate_distances3(arr):
    m = arr.shape[0]
    dist_arr = np.zeros((m, m))
    for i in range(m):
        for j in range(i):
            dist_arr[i, j] = np.linalg.norm(arr[i] - arr[j], 1)
            dist_arr[j, i] = dist_arr[i, j]
    return dist_arr
# 270 ms ± 12.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


@njit('float64[:,::1](float64[:,::1])', cache=True)
def calculate_distances4(arr):
    m = arr.shape[0]
    dist_arr = np.zeros((m, m))
    for i in range(m):
        for j in range(i):
            dist_arr[i, j] = np.linalg.norm(arr[i] - arr[j], 1)
            dist_arr[j, i] = dist_arr[i, j]
    return dist_arr
# 264 ms ± 2.21 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


@njit('float64[:,::1](float64[:,::1])', cache=True)
def calculate_distances5(arr):
    m = arr.shape[0]
    n = arr.shape[1]
    dist_arr = np.zeros((m, m))
    for i in range(m):
        for j in range(i):
            v = 0.0
            for k in range(n):
                v += abs(arr[i, k] - arr[j, k])
            dist_arr[i, j] = v
            dist_arr[j, i] = v
    return dist_arr
# 88.5 ms ± 2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


@njit('float64[:,::1](float64[:,::1])', cache=True, parallel=True, nogil=True)
def calculate_distances6(arr):
    m = arr.shape[0]
    n = arr.shape[1]
    dist_arr = np.zeros((m, m))
    for i in prange(m):
        for j in range(i):
            v = 0.0
            for k in range(n):
                v += abs(arr[i, k] - arr[j, k])
            dist_arr[i, j] = v
            dist_arr[j, i] = v
    return dist_arr
# 122 ms ± 10.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
# 88.5 ms ± 2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

In [10]:
%%timeit -r 3 -n 7
d1 = calculate_distances6(arr)

121 ms ± 2.22 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [25]:
%%timeit -r 3 -n 7
dz = dg.explode('ts')

691 ms ± 7.47 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [35]:
%%timeit -r 3 -n 7
d1 = df.copy()
d1['ts'] = [
        pd.date_range(start, end, freq='30min')
        for start, end in zip(df['start_date'].values, df['end_date'].values)
    ]
# d1 = d1.explode('ts')

684 ms ± 12.6 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [21]:
%%timeit -r 3 -n 7
for start, end in zip(df['start_date'].values, df['end_date'].values): pass

10.4 µs ± 2.55 µs per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [22]:
351/10.4

33.75

In [77]:
def explode_old(df, start_date_col, end_date_col, freq):
    t0 = time.time()
    df['ts'] = [
        pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
        for (_, row) in df.iterrows()
    ]
    print(f'Old  create list time: {time.time()-t0:.3f}')

    t0 = time.time()
    df = df.explode('ts')
    print(f'Old explode list time: {time.time()-t0:.3f}')
    return df

def explode_new(df, start_date_col, end_date_col, freq):
    # Get exploded timestamp column
    # t0 = time.time()
    dt = pd.concat([
        pd.DataFrame({'i': i, 'ts': pd.date_range(start=s, end=e, freq=freq)})
        for i, (s, e) in enumerate(zip(df[start_date_col], df[end_date_col]))
    ]).set_index('i').rename_axis(None, axis=0)
    # print(f'New  create list time: {time.time()-t0:.3f}')

    # Re-sampling df based on new timestamp column
    # t0 = time.time()
    df = df.reindex(dt.index).assign(ts=dt.ts)
    # print(f'New      reindex time: {time.time()-t0:.3f}')
    return df

In [31]:
start_date_col = 'start_date'
end_date_col = 'end_date'
freq = '30min'

In [78]:
%%timeit -r 8 -n 10
# t0 = time.time()
d2 = explode_new(df, 'start_date', 'end_date', '30min')
# t2 = time.time() - t0
# print(f'New time {t2:.3f}')

52.3 ms ± 555 µs per loop (mean ± std. dev. of 8 runs, 10 loops each)


In [33]:
t0 = time.time()
d1 = explode_old(df, 'start_date', 'end_date', '30min')
t1 = time.time() - t0
print(f'Old time {t1:.3f}')

Old  create list time: 0.962
Old explode list time: 1.018
Old time 1.981


In [7]:
%%time
df['ts'] = [
    pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
    for (_, row) in df.iterrows()
]

CPU times: user 6.26 s, sys: 131 ms, total: 6.4 s
Wall time: 6.4 s


In [8]:
%%time
d = df.explode('ts')

CPU times: user 6.38 s, sys: 224 ms, total: 6.6 s
Wall time: 6.6 s


In [9]:
df[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,"DatetimeIndex(['2022-07-01 00:00:00', '2022-07..."
1,ihXEKLSb,4Q2WJ5DjIMkiPBDWQ3kV,2022-06-01,2023-01-01,36.413329,78.649345,"DatetimeIndex(['2022-06-01 00:00:00', '2022-06..."


In [10]:
%%time
d = df.get(['ts']).reset_index(names=['id']).explode('ts')

CPU times: user 6.41 s, sys: 160 ms, total: 6.57 s
Wall time: 6.61 s


In [89]:
# %%time
d = df.get(['ts']).reset_index(drop=True).rename_axis('i', axis=0).reset_index()
d[:2]

,i,ts
0,0,"DatetimeIndex(['2020-01-01 00:00:00', '2020-01..."
1,1,"DatetimeIndex(['2020-05-01 00:00:00', '2020-05..."


In [83]:
# %%timeit -r 10 -n 100
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': row['ts']})
    for (key, row) in d.iterrows()
]).set_index('i').rename_axis(None, axis=0)
dt[:2]

,ts
0,2020-01-01 00:00:00
0,2020-01-01 00:30:00


In [50]:
%%timeit -r 10 -n 100
dt = pd.concat([
    pd.DataFrame({'i': i, 'ts': ts})
    for i, ts in zip(d['i'].values, d['ts'].values)
]).set_index('i').rename_axis(None, axis=0)
# dt[:2]

6.41 ms ± 181 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [13]:
%%time
d = df.drop(columns='ts').reindex(dt.index)
d['ts'] = dt.ts
d[:2]

CPU times: user 103 ms, sys: 32 ms, total: 135 ms
Wall time: 133 ms


,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:00:00
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:30:00


In [84]:
# %%timeit
# explode `ts` column
d = df.get(['ts']).reset_index(drop=True)
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': row['ts']})
    for (key, row) in d.iterrows()
]).set_index('i').rename_axis(None, axis=0)
d = df.drop(columns='ts').reindex(dt.index)
d['ts'] = dt.ts
d[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,jIMki,2020-01-01,2023-07-01,35.20661,76.041564,2020-01-01 00:00:00
0,8v5KSoKX,jIMki,2020-01-01,2023-07-01,35.20661,76.041564,2020-01-01 00:30:00


In [15]:
# %%timeit
# explode `ts` column and expand
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': pd.date_range(start=row['start_date'], end=row['end_date'], freq='30min')})
    for (key, row) in df.iterrows()
]).set_index('i').rename_axis(None, axis=0)
dx = df.drop(columns='ts').reindex(dt.index)
dx['ts'] = dt.ts
dx[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:00:00
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:30:00


In [90]:
def explode_df_column(df):
    d = df.get(['ts']).reset_index(drop=True).rename_axis('i', axis=0).reset_index()
    dt = pd.concat([
        pd.DataFrame({'i': i, 'ts': ts})
        for (i, ts) in zip(d['i'].values, d['ts'].values)
    ]).set_index('i').rename_axis(None, axis=0)
    df = df.drop(columns='ts').reindex(dt.index)
    df['ts'] = dt.ts
    return df
dd = df.copy()
dd['ts'] = [
    pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
    for (_, row) in dd.iterrows()
]

In [91]:
# %%timeit -r 10 -n 100
dx = dd.copy()
dy = explode_df_column(dx)

In [92]:
dz = dd.explode('ts')
dz.shape

(554986, 7)

In [93]:
dy.equals(dz)

True

In [58]:
%%timeit -r 10 -n 100
# Create a DataFrame with new index and exploded ts column
dt = pd.concat([
    pd.DataFrame({'i': i, 'ts': pd.date_range(start, end, freq='30min')})
    for i, (start, end) in enumerate(zip(df['start_date'], df['end_date']))
]).set_index('i').rename_axis(None, axis=0)

# Resample original df based on new index and add the exploded ts column
dn = df.reindex(dt.index).assign(ts=dt.ts)

49.8 ms ± 932 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [60]:
(703+691)/49.8

27.991967871485944